In [13]:
from pathlib import Path
import pandas as pd
import numpy as np
import sys

repo_root = Path.cwd().parent if Path.cwd().name == "playground" else Path.cwd()
sys.path.insert(0, str(repo_root))

from data_generation.ena_weather import (
    ENADataset,
    PAPER_TEST_MONTHS,
    PAPER_TEST_YEAR,
    _matlab_datenum_to_datetime,
    load_ena_supervised_dataset,
    select_real_split,
)


In [2]:
dataset: ENADataset = load_ena_supervised_dataset()

In [3]:
len(dataset.feature_names)

22

In [4]:
dataset.x.shape

(60866, 241, 22)

In [5]:
dataset.y

array([101.09395 , 187.16336 , 126.841125, ..., 116.82364 , 131.3857  ,
        98.990005], dtype=float32)

## Check whether `dataset.y` is already log-transformed

The paper trains on `log10(N_CCN)`. The loader does not apply a log transform, so this cell checks whether the `.mat` file already stores CCN on a log scale or on the original raw scale.

In [3]:
import numpy as np

y = np.asarray(dataset.y, dtype=float)
finite_y = y[np.isfinite(y)]
positive_y = finite_y[finite_y > 0]

levels = [0, 1, 5, 50, 95, 99, 100]
print("dataset.y percentiles [min, p01, p05, median, p95, p99, max]:")
print(np.round(np.nanpercentile(finite_y, levels), 4))

looks_log10 = np.nanmax(finite_y) < 10 and np.nanmedian(finite_y) < 5
print()
if looks_log10:
    print("Interpretation: dataset.y is probably already log10(CCN).")
    print("If y is log10(CCN), implied raw CCN percentiles are:")
    print(np.round(np.nanpercentile(10 ** finite_y, [1, 5, 50, 95, 99]), 4))
else:
    print("Interpretation: dataset.y looks like raw CCN, not log10(CCN).")
    print("If y is raw CCN, log10(CCN) percentiles are:")
    print(np.round(np.nanpercentile(np.log10(positive_y), [1, 5, 50, 95, 99]), 4))

dataset.y percentiles [min, p01, p05, median, p95, p99, max]:
[   9.4608   23.4737   42.7212  139.6417  353.9957  565.7149 1311.4496]

Interpretation: dataset.y looks like raw CCN, not log10(CCN).
If y is raw CCN, log10(CCN) percentiles are:
[1.3706 1.6306 2.145  2.549  2.7526]


## Check the paper split with timestamps

This verifies that the paper-style split uses only Jan/Mar/May/Jul/Sep/Nov 2022 as test months, keeps those months out of training, and creates disjoint train/test index sets.

In [8]:
train_idx, test_idx = select_real_split(
    dataset=dataset,
    split="paper",
    train_size=None,
    test_size=None,
    seed=2026 + 17,
)

dates = np.array([_matlab_datenum_to_datetime(t) for t in dataset.time], dtype=object)
train_dates = dates[train_idx]
test_dates = dates[test_idx]

paper_months = {(PAPER_TEST_YEAR, month) for month in PAPER_TEST_MONTHS}
train_months = {(d.year, d.month) for d in train_dates}
test_months = {(d.year, d.month) for d in test_dates}

print("train date range:", min(train_dates), "to", max(train_dates))
print("test date range:", min(test_dates), "to", max(test_dates))
print("paper held-out months:", sorted(paper_months))
print("test months present:", sorted(test_months))
print("held-out months missing from loaded data:", sorted(paper_months - test_months))
print("paper held-out months present in train:", sorted(train_months & paper_months))

assert len(set(train_idx).intersection(test_idx)) == 0
assert test_months <= paper_months
assert not (train_months & paper_months)

train date range: 2013-10-04 08:00:00.000003 to 2024-10-22 23:00:00.000003
test date range: 2022-01-01 00:00:00 to 2022-11-30 23:00:00.000003
paper held-out months: [(2022, 1), (2022, 3), (2022, 5), (2022, 7), (2022, 9), (2022, 11)]
test months present: [(2022, 1), (2022, 3), (2022, 5), (2022, 7), (2022, 9), (2022, 11)]
held-out months missing from loaded data: []
paper held-out months present in train: []


In [18]:
np.count_nonzero(dataset.flags.get('BB_criterion1') == True) # This many wildfires

6786